In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px  # interactive figures
import plotly.graph_objects as go
import seaborn as sns


In [2]:
import pandas as pd

df = pd.read_csv("../data_raw/date_24_art_loreal_20%.csv")
df = df[df["ID_ARTICOL"] == 147807]


df.head()


,DATA,ID_ARTICOL,ARTICOL,CANTITATE,VAL_IESIRE_FARA_TVA,VAL_IESIRE_CU_TVA,VAL_INTRARE_FARA_TVA,TOTAL_DISCOUNT,RATA_DISCOUNT,ADAOS,STOC_INITIAL,STOC_FINAL,RUPTURA_STOC
12,01-Jan-24,147807,LRP CICAPLAST B5+ BALSAM REPARATOR CALMANT CU ...,0,0.00,0.0,0.00,0.0,0.0,0.00,278,278,0
19,02-Jan-24,147807,LRP CICAPLAST B5+ BALSAM REPARATOR CALMANT CU ...,0,0.00,0.0,0.00,0.0,0.0,0.00,278,278,0
30,03-Jan-24,147807,LRP CICAPLAST B5+ BALSAM REPARATOR CALMANT CU ...,11,647.00,770.0,574.75,0.0,0.0,72.25,278,267,0
51,04-Jan-24,147807,LRP CICAPLAST B5+ BALSAM REPARATOR CALMANT CU ...,11,647.04,770.0,574.75,0.0,0.0,72.29,267,376,0
64,05-Jan-24,147807,LRP CICAPLAST B5+ BALSAM REPARATOR CALMANT CU ...,10,588.22,700.0,522.50,0.0,0.0,65.72,376,366,0


standard deviation is a way to measure how "spread out" or "different" the values in a group are from the average.

If the standard deviation is small, it means most values are close to the average (not much variation).
If the standard deviation is large, it means the values are spread out and vary a lot from the average.

Conversie coloana data din object in datetime

In [3]:
df['DATA'] = pd.to_datetime(df['DATA'], errors="coerce")

C:\Users\Mara\AppData\Local\Temp\ipykernel_23780\2340964975.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['DATA'] = pd.to_datetime(df['DATA'], errors="coerce")


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 735 entries, 12 to 9994
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   DATA                  735 non-null    datetime64[ns]
 1   ID_ARTICOL            735 non-null    int64         
 2   ARTICOL               735 non-null    object        
 3   CANTITATE             735 non-null    int64         
 4   VAL_IESIRE_FARA_TVA   735 non-null    float64       
 5   VAL_IESIRE_CU_TVA     735 non-null    float64       
 6   VAL_INTRARE_FARA_TVA  735 non-null    float64       
 7   TOTAL_DISCOUNT        735 non-null    float64       
 8   RATA_DISCOUNT         704 non-null    float64       
 9   ADAOS                 735 non-null    float64       
 10  STOC_INITIAL          735 non-null    int64         
 11  STOC_FINAL            735 non-null    int64         
 12  RUPTURA_STOC          735 non-null    int64         
dtypes: datetime64[ns](1), f

In [5]:
# Sortare logică pentru time series
df = df.sort_values(by=["DATA"]).reset_index(drop=True)

In [6]:
# reguli de calitate
checks = {}

# 1) vânzare > 0 dar valoare = 0 (suspect)
checks["qty_pos_val_zero"] = df[(df["CANTITATE"] > 0) & (df["VAL_IESIRE_CU_TVA"] == 0)]

# 2) valoare > 0 dar cantitate = 0 (suspect)
checks["val_pos_qty_zero"] = df[(df["VAL_IESIRE_CU_TVA"] > 0) & (df["CANTITATE"] == 0)]

# 3) stoc final negativ (dacă există în dataset)
if "STOC_FINAL" in df.columns:
    checks["stoc_final_negativ"] = df[df["STOC_FINAL"] < 0]

# 4) discount > 0 dar rata_discount = 0 (suspect)
if "TOTAL_DISCOUNT" in df.columns and "RATA_DISCOUNT" in df.columns:
    checks["disc_pos_rata_zero"] = df[(df["TOTAL_DISCOUNT"] > 0) & (df["RATA_DISCOUNT"] == 0)]

# 5) 
checks["stockout_start"] = df[(df["STOC_INITIAL"] == 0) & (df["CANTITATE"] == 0)]
checks["no_sales_but_in_stock"] = df[(df["STOC_INITIAL"] > 0) & (df["CANTITATE"] == 0)]

{k: v.shape[0] for k, v in checks.items()}

{'qty_pos_val_zero': 0,
 'val_pos_qty_zero': 0,
 'stoc_final_negativ': 0,
 'disc_pos_rata_zero': 0,
 'stockout_start': 0,
 'no_sales_but_in_stock': 16}

In [7]:
cols = ["CANTITATE", "STOC_INITIAL", "STOC_FINAL", "VAL_IESIRE_FARA_TVA"]

(df[cols] < 0).sum()


CANTITATE              0
STOC_INITIAL           0
STOC_FINAL             0
VAL_IESIRE_FARA_TVA    0
dtype: int64

In [8]:
# df = df.set_index('DATA') 

In [9]:
# df

Pentru evitarea redundanței informaționale, variabilele puternic corelate vor fi filtrate.  ( gen val iesire cu fara tva, total discount - rata discount )

In [10]:
df_feat = df.drop(columns=[ "STOC_FINAL", "RUPTURA_STOC", "VAL_IESIRE_CU_TVA", "RATA_DISCOUNT", "ARTICOL"])

df_feat


,DATA,ID_ARTICOL,CANTITATE,VAL_IESIRE_FARA_TVA,VAL_INTRARE_FARA_TVA,TOTAL_DISCOUNT,ADAOS,STOC_INITIAL
0,2024-01-01,147807,0,0.00,0.00,0.0,0.00,278
1,2024-01-02,147807,0,0.00,0.00,0.0,0.00,278
2,2024-01-03,147807,11,647.00,574.75,0.0,72.25,278
3,2024-01-04,147807,11,647.04,574.75,0.0,72.29,267
4,2024-01-05,147807,10,588.22,522.50,0.0,65.72,376
...,...,...,...,...,...,...,...,...
730,2025-12-31,147807,7,419.44,365.75,0.0,53.69,1429
731,2026-01-01,147807,0,0.00,0.00,0.0,0.00,1422
732,2026-01-02,147807,0,0.00,0.00,0.0,0.00,1422
733,2026-01-03,147807,13,778.96,679.25,0.0,99.71,1422


## PROPHET

In [11]:
daily = df.groupby("DATA", as_index=False)["CANTITATE"].sum()

# completează zilele lipsă
daily = daily.set_index("DATA").asfreq("D").fillna(0).reset_index()

# format Prophet: ds, y
prophet_df = daily.rename(columns={"DATA": "ds", "CANTITATE": "y"})
prophet_df["y"] = prophet_df["y"].astype(float)

# 2) split time-based
split_idx = len(prophet_df) - 14
train_df = prophet_df.iloc[:split_idx].copy()
test_df  = prophet_df.iloc[split_idx:].copy()

In [12]:
from prophet import Prophet

m = Prophet(
    growth="linear",
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    seasonality_mode="additive"  # poți încerca și "multiplicative"
)

# opțional: extra sezonalitate (dacă vrei să testezi)
# m.add_seasonality(name="monthly", period=30.5, fourier_order=5)

m.fit(train_df)


17:13:57 - cmdstanpy - INFO - Chain [1] start processing


17:13:57 - cmdstanpy - INFO - Chain [1] done processing


In [13]:
future = test_df[["ds"]].copy()
forecast = m.predict(future)

# yhat = predicția
pred = forecast["yhat"].values
y_true = test_df["y"].values

pred


array([ 9.43936389,  9.80522405, 10.56639873, 10.77971438, 13.16448343,
       14.44309841, 15.19472767, 11.50546596, 11.3080657 , 11.48247028,
       11.10299103, 12.90580942, 13.62917135, 13.86672205])

In [14]:
pred = np.maximum(pred, 0)


In [15]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


mae = mean_absolute_error(y_true, pred)
rmse = np.sqrt(mean_squared_error(y_true, pred))
r2 = r2_score(y_true, pred)

print("Prophet | MAE:", mae)
print("Prophet | RMSE:", rmse)
print("Prophet | R²:", r2)


Prophet | MAE: 7.283366204537068
Prophet | RMSE: 8.383706669667673
Prophet | R²: -1.7613071466253607


In [16]:
import plotly.express as px

df_plot = pd.DataFrame({
    "DATA": test_df["ds"],
    "Actual": y_true,
    "Pred_Prophet": pred
})

df_long = df_plot.melt("DATA", var_name="Serie", value_name="Cantitate")

fig = px.line(
    df_long,
    x="DATA",
    y="Cantitate",
    color="Serie",
    title="Prophet – Actual vs Predicted (test)"
)

fig.update_layout(
    hovermode="x unified",
    xaxis_title="Data",
    yaxis_title="Cantitate"
)

fig.update_xaxes(rangeslider_visible=True)
fig.show()
